# 🏆 GPT-SoVITS — Full Train (Colab)

Tomar **5-10 min recording** diye nijer voice model **train** korbe → highest quality clone (zero-shot er cheye onek bhalo, especially Bangla + English mixed).

**Age koro:** `Runtime` → `Change runtime type` → **T4 GPU** → Save.

### Pipeline (ei notebook e)
1. GPU check
2. Install GPT-SoVITS + pretrained models
3. Tomar audio upload (5-10 min)
4. **Slice** — boro audio choto choto clip e katbe
5. **ASR** — clip gulor text banabe (faster-whisper, Bangla+English supports)
6. **Train** — SoVITS + GPT model (WebUI diye, sohoj)
7. **Inference** — trained voice diye text bolabe

> Train + preprocess er sohoj-tomo poth holo **built-in WebUI** (GUI). Ei notebook WebUI ke ekta public link diye khule dey — oikhane click kore sob step korbe. Manual clip/ASR cell-o deya ache.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU nai! Runtime > Change runtime type > T4 GPU koro.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Install GPT-SoVITS + pretrained models
~5-8 min lagbe (repo + dependencies + pretrained weights download).

In [ ]:
%cd /content
!git clone https://github.com/RVC-Boss/GPT-SoVITS
%cd /content/GPT-SoVITS
!pip -q install -r requirements.txt
!pip -q install faster-whisper
print('Dependencies installed ✓')

In [ ]:
# Pretrained models download (HuggingFace theke). GPT-SoVITS eder chhara train hobe na.
%cd /content/GPT-SoVITS
!pip -q install "huggingface_hub[cli]"
from huggingface_hub import snapshot_download
snapshot_download(repo_id='lj1995/GPT-SoVITS',
                  local_dir='GPT_SoVITS/pretrained_models',
                  local_dir_use_symlinks=False)
print('Pretrained models ready ✓')
import os
print(os.listdir('GPT_SoVITS/pretrained_models')[:20])

## 3. Tomar audio upload
Tomar 5-10 min recording (`sample.wav`) upload koro (RECORDING_GUIDE.md follow kore banano).

In [ ]:
from google.colab import files
import os
os.makedirs('/content/GPT-SoVITS/input_raw', exist_ok=True)
print('Tomar full recording (wav/mp3) upload koro...')
up = files.upload()
RAW = '/content/GPT-SoVITS/input_raw/' + list(up.keys())[0]
os.replace(list(up.keys())[0], RAW)
print('Uploaded:', RAW)

## 4. Slice — boro audio -> choto clip
Model train er jonno 3-10 sec er choto clip lage. Ei cell auto slice korbe.

In [ ]:
%cd /content/GPT-SoVITS
import os
os.makedirs('/content/GPT-SoVITS/input_sliced', exist_ok=True)
# GPT-SoVITS er built-in slicer (audio_sr, dir_out, threshold, min_len, ...)
!python tools/slice_audio.py "{RAW}" "/content/GPT-SoVITS/input_sliced" -34 4000 300 10 500 0.9 0.25 0 1
print('\nSliced clips:')
print(sorted(os.listdir('/content/GPT-SoVITS/input_sliced'))[:20])

## 5. ASR — clip gulor text (transcription)
faster-whisper (large-v3) diye clip gulor text banabe. **Bangla + English dutai supports.**
Ekta `.list` file toiri hobe ja GPT-SoVITS training e lage.

> Bangla er jonno `language='bn'`, English hole `'en'`, mixed hole `None` (auto-detect).

In [ ]:
import os
from faster_whisper import WhisperModel

SLICED = '/content/GPT-SoVITS/input_sliced'
LIST_PATH = '/content/GPT-SoVITS/input_sliced.list'
SPK = 'myvoice'
LANG = None   # 'bn' = Bangla, 'en' = English, None = auto-detect (mixed er jonno)

model = WhisperModel('large-v3', device='cuda', compute_type='float16')
lines = []
for fn in sorted(os.listdir(SLICED)):
    if not fn.lower().endswith('.wav'):
        continue
    fp = os.path.join(SLICED, fn)
    segments, info = model.transcribe(fp, language=LANG, beam_size=5)
    text = ''.join(s.text for s in segments).strip()
    lang = (LANG or info.language)
    lang_tag = {'bn': 'bn', 'en': 'en'}.get(lang, 'en')
    if text:
        # GPT-SoVITS .list format:  wavpath|speaker|lang|text
        lines.append(f'{fp}|{SPK}|{lang_tag}|{text}')
        print(f'[{lang_tag}] {fn}: {text[:60]}')

with open(LIST_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'\n✓ {len(lines)} clips transcribed -> {LIST_PATH}')

## 6. Train — WebUI diye (recommended)
Ei cell WebUI ke ekta **public link** diye khule dey. Link e click kore GUI te train koro:

**WebUI te steps:**
1. **1-GPT-SoVITS-TTS → Formatting**: audio dir = `input_sliced`, list file = `input_sliced.list` diye `1a/1b/1c` (One-click) run koro.
2. **1B-Fine-tuned training**: `1Ba` (SoVITS train) + `1Bb` (GPT train) run koro. Epochs default rakho (SoVITS ~8, GPT ~15). T4 te ~15-30 min.
3. **1C-inference**: `Open TTS Inference WebUI` → trained model select → reference audio + text diye generate.

> Link (trycloudflare) ekta cell output e ashbe — `https://...trycloudflare.com`.

In [ ]:
%cd /content/GPT-SoVITS
# Public tunnel (cloudflared) — Colab e WebUI kholar jonno
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
import subprocess, threading, time, re

# WebUI background e chalu (default port 9874)
def run_webui():
    subprocess.Popen(['python', 'webui.py'], cwd='/content/GPT-SoVITS')
run_webui()
print('WebUI starting... 20 sec wait...')
time.sleep(20)

# Tunnel
proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:9874'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n🔗 WebUI LINK:', m.group(0))
        break

## 7. Trained model download (backup)
Train shesh hole model weights `SoVITS_weights*/` ও `GPT_weights*/` folder e thake. Download kore rakho (Colab session shesh hole muche jabe):

In [ ]:
%cd /content/GPT-SoVITS
!zip -r -q /content/my_voice_model.zip SoVITS_weights* GPT_weights* 2>/dev/null || echo 'weights folder ekhono nai — age train koro'
import os
if os.path.exists('/content/my_voice_model.zip'):
    from google.colab import files
    files.download('/content/my_voice_model.zip')